# Week 07 · NLP Take-Home Assignment
## TF-IDF, Sentiment, & Embeddings
This notebook demonstrates solutions to Q1 and Q2 using modular python architecture.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from src.data_generator import create_shopsense_reviews
from src.tfidf_engine import compute_corpus_stats, build_tfidf_matrix, compute_cosine_similarity
from src.analytical_models import compute_manual_tfidf, compute_bm25_scores
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse.linalg import norm as spnsnorm

# Generated if not exists
try:
    df = pd.read_csv('../data/ShopSense_Reviews.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Dataset not found. Generating...")
    df = create_shopsense_reviews('../data/ShopSense_Reviews.csv')

print(f"Dataset shape: {df.shape}")
df.head(3)

Dataset loaded successfully.
Dataset shape: (10000, 6)


,review_id,customer_id,product_id,category,review_text,rating
0,0,2424,704,Books,good okay okay very very item the,4
1,1,1106,877,Electronics,earbuds battery sound charging screen screen e...,2
2,2,7201,180,Books,the bad item the good good product a product v...,5


## Q1 (a): TF-IDF Matrix from Scratch

In [2]:
docs = df['review_text'].fillna('').tolist()
tf_list, df_counts, idf = compute_corpus_stats(docs)

vocab = sorted(list(idf.keys()))
vocab_index = {word: i for i, word in enumerate(vocab)}

tfidf_matrix = build_tfidf_matrix(tf_list, idf, vocab)
print(f"Vocabulary size: {len(vocab)}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

2026-04-12 20:36:16,766 - INFO - Computing corpus term frequencies and document frequencies...


2026-04-12 20:36:16,795 - INFO - Building sparse TF-IDF matrix...


Vocabulary size: 31
TF-IDF matrix shape: (10000, 31)


## Q1 (b): Ranking Query with Cosine Similarity

In [3]:
query = 'wireless earbuds battery life poor'
cos_sim = compute_cosine_similarity(query, tfidf_matrix, idf, vocab_index)

top_5_indices = np.argsort(cos_sim)[::-1][:5]
print("Top 5 Results for Cosine:")
for idx in top_5_indices:
    print(f"[{idx}] Score: {cos_sim[idx]:.4f} | {df.iloc[idx]['review_text']}")

Top 5 Results for Cosine:
[3016] Score: 0.9409 | earbuds poor life wireless battery poor wireless the
[4368] Score: 0.9308 | earbuds life wireless very battery battery poor the
[2268] Score: 0.9304 | earbuds life wireless the poor is battery very the
[391] Score: 0.9299 | earbuds earbuds is life poor battery battery wireless the
[2548] Score: 0.9137 | earbuds the battery earbuds excellent wireless life life the wireless battery life poor the


## Q1 (c): Sklearn Comparison

In [4]:
vectorizer_custom = TfidfVectorizer(tokenizer=lambda x: x.lower().split(), token_pattern=None, lowercase=False)
sklearn_custom_tfidf = vectorizer_custom.fit_transform(docs)

diff = tfidf_matrix - sklearn_custom_tfidf
avg_l2_diff = spnsnorm(diff) / len(docs)
print(f"Average L2 difference between custom and sklearn TF-IDF: {avg_l2_diff:.8f}")

Average L2 difference between custom and sklearn TF-IDF: 0.00000000


## Q1 (d): Electronics Category TF-IDF

In [5]:
electronics_idx = df[df['category'] == 'Electronics'].index
electronics_matrix = tfidf_matrix[electronics_idx]
avg_tfidf_electronics = np.array(electronics_matrix.mean(axis=0))[0]

max_idx = np.argmax(avg_tfidf_electronics)
print(f"Top word in Electronics: {vocab[max_idx]}")
print(f"Average Output TF-IDF: {avg_tfidf_electronics[max_idx]:.4f}")

Top word in Electronics: earbuds
Average Output TF-IDF: 0.3360


### Reason:
The word 'earbuds' ranks at the top because it has high term frequency in Electronics, but a very high global IDF since it doesn't appear in Clothing, Food, etc.

## Q2 (a): Manual TF-IDF for 'fabric'

In [6]:
doc_42_text = df.loc[42, 'review_text']
print(f"Doc_42: {doc_42_text}")

tokens_42 = doc_42_text.lower().split()
stats = compute_manual_tfidf('fabric', tokens_42, len(docs), df_counts.get('fabric', 0))

print(f"TF: {stats['tf']}")
print(f"IDF: {stats['idf']:.4f}")
print(f"Raw TF-IDF ('fabric'): {stats['tfidf_raw']:.4f}")

Doc_42: the fabric of this is great and the embroidery is nice
TF: 1
IDF: 3.4351
Raw TF-IDF ('fabric'): 3.4351


## Q2 (b): IDF('the') vs IDF('embroidery')

In [7]:
for w in ['the', 'embroidery']:
    stat = compute_manual_tfidf(w, [], len(docs), df_counts.get(w, 0))
    print(f"IDF({w}) = {stat['idf']:.4f}")

IDF(the) = 1.0000
IDF(embroidery) = 3.4260


**Explanation:**
'the' is a stop word appearing in almost every document, pushing DF towards N and making `ln(1)` = 0. 'embroidery' is rare, making `DF` tiny, giving it a massive IDF weight.

## Q2 (c): Rebuttal
Simply using word frequency over-rewards common stop words like 'the' or 'is' which carry little semantic meaning. TF-IDF elegantly discounts these ubiquitous terms by multiplying frequency with Inverse Document Frequency (IDF), highlighting terms that are frequent in a specific document but rare globally. This dual-balancing mechanism ensures that domain-specific keywords accurately represent document uniqueness instead of grammatical filler.

## Q2 (Bonus): BM25 Variant

In [8]:
bm25_scores = compute_bm25_scores(docs, query.split(), df_counts, len(docs))
top_5_bm25_idx = np.argsort(bm25_scores)[::-1][:5]

print("Top 5 Results using BM25:")
for idx in top_5_bm25_idx:
    print(f"[{idx}] Score: {bm25_scores[idx]:.4f} | {df.iloc[idx]['review_text']}")

Top 5 Results using BM25:
[3016] Score: 14.7840 | earbuds poor life wireless battery poor wireless the
[2548] Score: 14.2173 | earbuds the battery earbuds excellent wireless life life the wireless battery life poor the
[391] Score: 13.9699 | earbuds earbuds is life poor battery battery wireless the
[4508] Score: 13.9699 | earbuds battery battery poor wireless sound life earbuds the
[4368] Score: 13.7345 | earbuds life wireless very battery battery poor the
